# 02 Model Screening and Hyperparameter Tuning at 500 m

This notebook performs leakage-safe hyperparameter tuning for the two strongest
tabular machine-learning models:

- Random Forest
- XGBoost

Evaluation uses nested cross-validation under two validation schemes:

1. Random 5-fold cross-validation
2. KMeans-based spatial leave-one-cluster-out validation (K = 4)

Hyperparameter search is performed only within each outer training fold.
The outer test fold is never used for parameter selection.

In [ ]:
# ============================================================
# Imports and experiment settings
# ============================================================

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)
from sklearn.model_selection import (
    KFold,
    GroupKFold,
    RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
RANDOM_FOLDS = 5
SPATIAL_CLUSTERS = 4
INNER_FOLDS = 3

SEARCH_ITERATIONS = {
    "RandomForest": 30,
    "XGBoost": 30,
}

MODEL_ORDER = [
    "RandomForest",
    "XGBoost",
]

VERSION_ORDER = [
    "Untuned",
    "Tuned",
]

SET2_PALETTE = sns.color_palette("Set2", n_colors=4)

In [ ]:
# ============================================================
# Load the 500 m feature matrix
# ============================================================

import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from project_config import RAW_DATA_DIR, WORK_DIR

PROJECT_DATA_DIR = WORK_DIR

FEATURE_MATRIX_PATH = (
    PROJECT_DATA_DIR
    / "grid_size_selection"
    / "features"
    / "feature_matrix_500m.csv"
)


OUTPUT_DIR = (
    PROJECT_DATA_DIR
    / "model_tuning_500m"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TARGET_COLUMN = "elec_consumption"

BASELINE_FEATURES = [
    "area_m2",

    "B02_mean", "B02_std", "B02_max",
    "B03_mean", "B03_std", "B03_max",
    "B04_mean", "B04_std", "B04_max",
    "B08_mean", "B08_std", "B08_max",
    "B11_mean", "B11_std", "B11_max",

    "NDVI", "NDBI",

    "NTL_mean",  "NTL_max",

    "dist_major_road",
    "road_count",
    "road_length_major",
    "road_density",
    "major_road_ratio",

    "poi_economic_count",
    "poi_social_count",
    "poi_other_count",
    "poi_shannon",
    "dist_economic_poi",
    "dist_social_poi",

    "pop_density_km2",

    "building_count",
    "building_area_mean",
    "building_area_std",
    "building_coverage",
]

df = pd.read_csv(FEATURE_MATRIX_PATH)

df = (
    df
    .sort_values("grid_id")
    .reset_index(drop=True)
)

required_columns = (
    ["grid_id", "centroid_x", "centroid_y", TARGET_COLUMN]
    + BASELINE_FEATURES
)

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

print("Samples:", len(df))
print("Predictors:", len(BASELINE_FEATURES))
print("Target:", TARGET_COLUMN)

In [ ]:
# ============================================================
# Select the number of spatial clusters on the final 500 m grid
# ============================================================

from sklearn.metrics import silhouette_score

K_RANGE = range(3, 11)

coordinates_500m = df[
    ["centroid_x", "centroid_y"]
].to_numpy()

k_results = []

for k in K_RANGE:
    kmeans = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10,
    )

    labels = kmeans.fit_predict(
        coordinates_500m
    )

    cluster_sizes = np.bincount(labels)

    k_results.append({
        "k": k,
        "silhouette": silhouette_score(
            coordinates_500m,
            labels,
        ),
        "inertia": kmeans.inertia_,
        "min_cluster_size": cluster_sizes.min(),
        "max_cluster_size": cluster_sizes.max(),
        "balance_ratio": (
            cluster_sizes.min()
            / cluster_sizes.max()
        ),
    })

k_selection = pd.DataFrame(k_results)

display(
    k_selection.round(4)
)

In [ ]:
# ============================================================
# Visualise spatial K selection
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4),
)

sns.lineplot(
    data=k_selection,
    x="k",
    y="silhouette",
    marker="o",
    ax=axes[0],
)

axes[0].set_title("Silhouette Score")
axes[0].set_xlabel("Number of clusters (K)")
axes[0].set_ylabel("Silhouette score")

sns.lineplot(
    data=k_selection,
    x="k",
    y="balance_ratio",
    marker="o",
    ax=axes[1],
)

axes[1].set_title("Cluster Size Balance")
axes[1].set_xlabel("Number of clusters (K)")
axes[1].set_ylabel("Min / max cluster size")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Define final KMeans spatial folds on the 500 m grid
# ============================================================

coordinates_500m = df[
    ["centroid_x", "centroid_y"]
].to_numpy()

final_kmeans = KMeans(
    n_clusters=SPATIAL_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=10,
)

df["spatial_block"] = final_kmeans.fit_predict(
    coordinates_500m
).astype(int)

spatial_block_summary = (
    df.groupby("spatial_block")
    .agg(
        n_samples=("grid_id", "count"),
        target_mean=(TARGET_COLUMN, "mean"),
        target_std=(TARGET_COLUMN, "std"),
    )
    .reset_index()
)

display(spatial_block_summary.round(3))

df[
    [
        "grid_id",
        "centroid_x",
        "centroid_y",
        "spatial_block",
    ]
].to_csv(
    OUTPUT_DIR / "spatial_block_assignments_500m.csv",
    index=False,
)

print(
    df["spatial_block"]
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================================
# Inspect target distribution across spatial folds
# ============================================================

target_distribution = (
    df.groupby("spatial_block")[TARGET_COLUMN]
    .agg(
        count="count",
        mean="mean",
        std="std",
        median="median",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        p95=lambda x: x.quantile(0.95),
        max="max",
    )
    .reset_index()
)

target_distribution["cv"] = (
    target_distribution["std"]
    / target_distribution["mean"]
)

display(target_distribution.round(3))

plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x="spatial_block",
    y=TARGET_COLUMN,
    palette=SET2_PALETTE,
)

plt.xlabel("Spatial block")
plt.ylabel("Electricity consumption")
plt.title("Target Distribution across Spatial Validation Folds")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Map 1: Final KMeans spatial validation folds
# ============================================================

import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

GRID_500M_PATH = (
    PROJECT_DATA_DIR
    / "grid_size_selection"
    / "grids"
    / "grid_features_500m.gpkg"
)

grid_500m = gpd.read_file(GRID_500M_PATH)

grid_spatial = (
    grid_500m[["grid_id", "geometry"]]
    .merge(
        df[["grid_id", "spatial_block"]],
        on="grid_id",
        how="inner",
        validate="one_to_one",
    )
)

spatial_palette = sns.color_palette(
    "Set2",
    n_colors=SPATIAL_CLUSTERS,
)

spatial_cmap = mcolors.ListedColormap(
    spatial_palette
)

fig, ax = plt.subplots(figsize=(9, 8))

grid_spatial.plot(
    column="spatial_block",
    cmap=spatial_cmap,
    vmin=0,
    vmax=SPATIAL_CLUSTERS - 1,
    edgecolor="white",
    linewidth=0.15,
    ax=ax,
)

legend_handles = [
    mpatches.Patch(
        color=spatial_palette[i],
        label=f"Block {i} (n={(df['spatial_block'] == i).sum()})",
    )
    for i in range(SPATIAL_CLUSTERS)
]

ax.legend(
    handles=legend_handles,
    title="Spatial validation fold",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
)

ax.set_title(
    "Final KMeans Spatial Validation Folds at 500 m"
)

ax.set_axis_off()
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Map 2: Spatial validation folds within the Karachi boundary
# ============================================================

BOUNDARY_PATH = RAW_DATA_DIR / "boundary" / "karachi_boundary_mask.json"

karachi_boundary = (
    gpd.read_file(BOUNDARY_PATH)
    .to_crs(grid_spatial.crs)
    .dissolve()[["geometry"]]
)

fig, ax = plt.subplots(figsize=(9, 8))

# Plot the full Karachi boundary
karachi_boundary.plot(
    ax=ax,
    facecolor="whitesmoke",
    edgecolor="black",
    linewidth=0.8,
)

# Overlay labelled 500 m grids
grid_spatial.plot(
    column="spatial_block",
    cmap=spatial_cmap,
    vmin=0,
    vmax=SPATIAL_CLUSTERS - 1,
    edgecolor="white",
    linewidth=0.12,
    ax=ax,
)

ax.legend(
    handles=legend_handles,
    title="Spatial validation fold",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
)

ax.set_title(
    "Spatial Validation Folds within the Karachi Boundary"
)

ax.set_axis_off()
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Create fixed outer validation folds
# ============================================================

random_splitter = KFold(
    n_splits=RANDOM_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

random_outer_splits = [
    {
        "fold": fold,
        "split_id": fold,
        "train_index": train_index,
        "test_index": test_index,
    }
    for fold, (train_index, test_index)
    in enumerate(
        random_splitter.split(df),
        start=1,
    )
]

spatial_outer_splits = []

for block_id in sorted(
    df["spatial_block"].unique()
):
    test_index = np.flatnonzero(
        df["spatial_block"].to_numpy()
        == block_id
    )

    train_index = np.flatnonzero(
        df["spatial_block"].to_numpy()
        != block_id
    )

    spatial_outer_splits.append({
        "fold": block_id + 1,
        "split_id": int(block_id),
        "train_index": train_index,
        "test_index": test_index,
    })

VALIDATION_SCHEMES = {
    "random_5fold": random_outer_splits,
    "spatial_kmeans_leave_one_out": (
        spatial_outer_splits
    ),
}

for name, splits in VALIDATION_SCHEMES.items():
    print(
        name,
        "test sizes:",
        [
            len(split["test_index"])
            for split in splits
        ],
    )

In [ ]:
# ============================================================
# Define baseline models and search spaces
# ============================================================

BASE_MODELS = {
    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),

    "XGBoost": XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=6,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbosity=0,
    ),
}

SEARCH_SPACES = {
    "RandomForest": {
        "model__n_estimators": [
            300, 500, 800,
        ],
        "model__max_depth": [
            None, 8, 12, 16, 20,
        ],
        "model__min_samples_split": [
            2, 5, 10, 20,
        ],
        "model__min_samples_leaf": [
            1, 2, 4, 8,
        ],
        "model__max_features": [
            "sqrt", 0.5, 0.7, 1.0,
        ],
    },

    "XGBoost": {
        "model__n_estimators": [
            300, 500, 800, 1200,
        ],
        "model__learning_rate": [
            0.01, 0.03, 0.05, 0.08,
        ],
        "model__max_depth": [
            2, 3, 4, 5, 6,
        ],
        "model__min_child_weight": [
            1, 3, 5, 10,
        ],
        "model__subsample": [
            0.60, 0.75, 0.90, 1.00,
        ],
        "model__colsample_bytree": [
            0.50, 0.70, 0.90, 1.00,
        ],
        "model__reg_alpha": [
            0.0, 0.01, 0.1, 0.5, 1.0,
        ],
        "model__reg_lambda": [
            1.0, 3.0, 5.0, 10.0, 20.0,
        ],
        "model__gamma": [
            0.0, 0.05, 0.10, 0.25,
        ],
    },
}

In [ ]:
# ============================================================
# Nested cross-validation helpers
# ============================================================

def regression_metrics(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(
            mean_squared_error(
                y_true,
                y_pred,
            )
        ),
        "MAE": mean_absolute_error(
            y_true,
            y_pred,
        ),
    }


def make_pipeline(model):
    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "model",
            model,
        ),
    ])


def make_inner_cv(
    train_frame,
    validation_name,
    fold_seed,
):
    if validation_name == "random_5fold":
        return (
            KFold(
                n_splits=INNER_FOLDS,
                shuffle=True,
                random_state=fold_seed,
            ),
            None,
        )

    inner_groups = KMeans(
        n_clusters=INNER_FOLDS,
        random_state=fold_seed,
        n_init=10,
    ).fit_predict(
        train_frame[
            ["centroid_x", "centroid_y"]
        ].to_numpy()
    )

    return (
        GroupKFold(
            n_splits=INNER_FOLDS
        ),
        inner_groups,
    )


def evaluate_nested_model(
    frame,
    validation_name,
    outer_splits,
    model_name,
):
    fold_rows = []
    prediction_frames = []
    search_frames = []

    model_template = BASE_MODELS[
        model_name
    ]

    for split in outer_splits:
        fold = split["fold"]

        train_index = split["train_index"]
        test_index = split["test_index"]

        train_frame = frame.iloc[
            train_index
        ].copy()

        test_frame = frame.iloc[
            test_index
        ].copy()

        X_train = train_frame[
            BASELINE_FEATURES
        ]

        X_test = test_frame[
            BASELINE_FEATURES
        ]

        y_train = train_frame[
            TARGET_COLUMN
        ].to_numpy()

        y_test = test_frame[
            TARGET_COLUMN
        ].to_numpy()

        # Evaluate the original untuned model
        baseline_model = make_pipeline(
            clone(model_template)
        )

        baseline_model.fit(
            X_train,
            y_train,
        )

        baseline_test_pred = (
            baseline_model.predict(X_test)
        )

        baseline_train_pred = (
            baseline_model.predict(X_train)
        )

        baseline_metrics = regression_metrics(
            y_test,
            baseline_test_pred,
        )

        fold_rows.append({
            "validation": validation_name,
            "model": model_name,
            "version": "Untuned",
            "fold": fold,
            "split_id": split["split_id"],
            **baseline_metrics,
            "train_R2": r2_score(
                y_train,
                baseline_train_pred,
            ),
            "best_params": None,
        })

        prediction_frames.append(
            pd.DataFrame({
                "grid_id": (
                    test_frame["grid_id"]
                    .to_numpy()
                ),
                "validation": validation_name,
                "model": model_name,
                "version": "Untuned",
                "fold": fold,
                "observed": y_test,
                "predicted": baseline_test_pred,
            })
        )

        fold_seed = RANDOM_STATE + fold

        inner_cv, inner_groups = (
            make_inner_cv(
                train_frame,
                validation_name,
                fold_seed,
            )
        )

        search = RandomizedSearchCV(
            estimator=make_pipeline(
                clone(model_template)
            ),
            param_distributions=(
                SEARCH_SPACES[model_name]
            ),
            n_iter=(
                SEARCH_ITERATIONS[
                    model_name
                ]
            ),
            scoring="r2",
            refit=True,
            cv=inner_cv,
            random_state=fold_seed,
            n_jobs=-1,
            return_train_score=False,
        )

        if inner_groups is None:
            search.fit(
                X_train,
                y_train,
            )
        else:
            search.fit(
                X_train,
                y_train,
                groups=inner_groups,
            )

        tuned_model = search.best_estimator_

        tuned_test_pred = (
            tuned_model.predict(X_test)
        )

        tuned_train_pred = (
            tuned_model.predict(X_train)
        )

        tuned_metrics = regression_metrics(
            y_test,
            tuned_test_pred,
        )

        fold_rows.append({
            "validation": validation_name,
            "model": model_name,
            "version": "Tuned",
            "fold": fold,
            "split_id": split["split_id"],
            **tuned_metrics,
            "train_R2": r2_score(
                y_train,
                tuned_train_pred,
            ),
            "best_params": json.dumps(
                search.best_params_
            ),
        })

        prediction_frames.append(
            pd.DataFrame({
                "grid_id": (
                    test_frame["grid_id"]
                    .to_numpy()
                ),
                "validation": validation_name,
                "model": model_name,
                "version": "Tuned",
                "fold": fold,
                "observed": y_test,
                "predicted": tuned_test_pred,
            })
        )

        search_result = pd.DataFrame(
            search.cv_results_
        )

        search_result[
            "validation"
        ] = validation_name

        search_result["model"] = model_name
        search_result["outer_fold"] = fold

        search_frames.append(
            search_result
        )

        print(
            validation_name,
            "|",
            model_name,
            "| fold",
            fold,
            "| best inner R²:",
            round(
                search.best_score_,
                4,
            ),
        )

    return (
        pd.DataFrame(fold_rows),
        pd.concat(
            prediction_frames,
            ignore_index=True,
        ),
        pd.concat(
            search_frames,
            ignore_index=True,
        ),
    )

In [ ]:
# ============================================================
# Run nested hyperparameter tuning
# ============================================================

all_fold_results = []
all_predictions = []
all_search_results = []

for validation_name, outer_splits in (
    VALIDATION_SCHEMES.items()
):
    for model_name in MODEL_ORDER:
        print(
            "\n",
            "=" * 60,
        )
        print(
            validation_name,
            "|",
            model_name,
        )

        (
            fold_result,
            prediction_result,
            search_result,
        ) = evaluate_nested_model(
            frame=df,
            validation_name=validation_name,
            outer_splits=outer_splits,
            model_name=model_name,
        )

        all_fold_results.append(
            fold_result
        )

        all_predictions.append(
            prediction_result
        )

        all_search_results.append(
            search_result
        )

tuning_fold_results = pd.concat(
    all_fold_results,
    ignore_index=True,
)

tuning_predictions = pd.concat(
    all_predictions,
    ignore_index=True,
)

tuning_search_results = pd.concat(
    all_search_results,
    ignore_index=True,
)

print("\nNested tuning completed.")

In [ ]:
# ============================================================
# Summarise nested tuning performance
# ============================================================

fold_summary = (
    tuning_fold_results
    .groupby(
        [
            "validation",
            "model",
            "version",
        ],
        as_index=False,
    )
    .agg(
        n_folds=("fold", "count"),
        R2_mean=("R2", "mean"),
        R2_std=(
            "R2",
            lambda x: x.std(ddof=0),
        ),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=(
            "RMSE",
            lambda x: x.std(ddof=0),
        ),
        MAE_mean=("MAE", "mean"),
        MAE_std=(
            "MAE",
            lambda x: x.std(ddof=0),
        ),
        train_R2_mean=(
            "train_R2",
            "mean",
        ),
    )
)

oof_rows = []

for keys, group in tuning_predictions.groupby(
    [
        "validation",
        "model",
        "version",
    ]
):
    validation, model, version = keys

    metrics = regression_metrics(
        group["observed"],
        group["predicted"],
    )

    oof_rows.append({
        "validation": validation,
        "model": model,
        "version": version,
        "OOF_R2": metrics["R2"],
        "OOF_RMSE": metrics["RMSE"],
        "OOF_MAE": metrics["MAE"],
    })

oof_summary = pd.DataFrame(
    oof_rows
)

tuning_summary = fold_summary.merge(
    oof_summary,
    on=[
        "validation",
        "model",
        "version",
    ],
    how="left",
)

display(
    tuning_summary
    .sort_values(
        [
            "validation",
            "model",
            "version",
        ]
    )
    .round(4)
)

tuning_fold_results.to_csv(
    OUTPUT_DIR
    / "nested_tuning_fold_results.csv",
    index=False,
)

tuning_predictions.to_csv(
    OUTPUT_DIR
    / "nested_tuning_oof_predictions.csv",
    index=False,
)

tuning_search_results.to_csv(
    OUTPUT_DIR
    / "nested_tuning_search_results.csv",
    index=False,
)

tuning_summary.to_csv(
    OUTPUT_DIR
    / "nested_tuning_summary.csv",
    index=False,
)

In [ ]:
# ============================================================
# Inspect outer-fold test performance after tuning
# ============================================================

outer_fold_performance = (
    tuning_fold_results[
        tuning_fold_results["version"] == "Tuned"
    ][
        [
            "validation",
            "model",
            "fold",
            "R2",
            "RMSE",
            "MAE",
            "train_R2",
        ]
    ]
    .sort_values(
        ["validation", "model", "fold"]
    )
    .reset_index(drop=True)
)

display(
    outer_fold_performance.round(4)
)

In [ ]:
# ============================================================
# Plot tuned vs untuned model performance
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Prepare plotting data
# ------------------------------------------------------------

plot_df = tuning_summary.copy()

# Keep only the two models used in formal tuning
plot_df = plot_df[
    plot_df["model"].isin(
        ["RandomForest", "XGBoost"]
    )
].copy()

# Standardise labels for plotting
validation_labels = {
    "random_5fold": "Random CV",
    "spatial_kmeans_leave_one_out": "Spatial CV",
}

model_labels = {
    "RandomForest": "Random Forest",
    "XGBoost": "XGBoost",
}

plot_df["validation_label"] = (
    plot_df["validation"]
    .map(validation_labels)
)

plot_df["model_label"] = (
    plot_df["model"]
    .map(model_labels)
)

plot_df["version_label"] = (
    plot_df["version"]
    .astype(str)
    .str.lower()
    .map({
        "untuned": "Untuned",
        "tuned": "Tuned",
    })
)

# ------------------------------------------------------------
# Create ordered category labels
# ------------------------------------------------------------

category_order = [
    ("Random CV", "Random Forest"),
    ("Random CV", "XGBoost"),
    ("Spatial CV", "Random Forest"),
    ("Spatial CV", "XGBoost"),
]

x_labels = [
    f"{validation}\n{model}"
    for validation, model in category_order
]

untuned_values = []
tuned_values = []

for validation, model in category_order:

    subset = plot_df[
        (plot_df["validation_label"] == validation)
        & (plot_df["model_label"] == model)
    ]

    untuned_row = subset[
        subset["version_label"] == "Untuned"
    ]

    tuned_row = subset[
        subset["version_label"] == "Tuned"
    ]

    untuned_values.append(
        untuned_row["OOF_R2"].iloc[0]
        if len(untuned_row) > 0
        else np.nan
    )

    tuned_values.append(
        tuned_row["OOF_R2"].iloc[0]
        if len(tuned_row) > 0
        else np.nan
    )

# ------------------------------------------------------------
# Plot grouped bars
# ------------------------------------------------------------

x = np.arange(len(x_labels))
bar_width = 0.36

fig, ax = plt.subplots(
    figsize=(10, 6)
)

bars_untuned = ax.bar(
    x - bar_width / 2,
    untuned_values,
    width=bar_width,
    label="Untuned",
)

bars_tuned = ax.bar(
    x + bar_width / 2,
    tuned_values,
    width=bar_width,
    label="Tuned",
)

# Zero reference line
ax.axhline(
    0,
    linewidth=1,
)

# ------------------------------------------------------------
# Add value labels
# ------------------------------------------------------------

def add_bar_labels(bars):
    for bar in bars:
        height = bar.get_height()

        if np.isnan(height):
            continue

        offset = 0.015 if height >= 0 else -0.025
        va = "bottom" if height >= 0 else "top"

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + offset,
            f"{height:.3f}",
            ha="center",
            va=va,
            fontsize=9,
        )


add_bar_labels(bars_untuned)
add_bar_labels(bars_tuned)

# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

ax.set_xticks(x)
ax.set_xticklabels(
    x_labels,
    fontsize=10,
)

ax.set_ylabel(
    "OOF $R^2$",
    fontsize=11,
)

ax.set_xlabel(
    "",
)

ax.set_title(
    "Effect of Hyperparameter Tuning on Model Performance",
    fontsize=13,
)

ax.legend(
    frameon=False,
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# ------------------------------------------------------------
# Save figure
# ------------------------------------------------------------

figure_path = (
    OUTPUT_DIR
    / "tuned_vs_untuned_model_performance.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(
    f"Saved figure to: {figure_path}"
)

In [ ]:
# ============================================================
# Compare untuned vs tuned RF and XGBoost
# Mean outer-fold R² ± SD
# Custom colours:
# RF untuned = light blue
# RF tuned   = blue
# XGB untuned = light orange
# XGB tuned   = orange
# ============================================================

import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Prepare plotting data
# ------------------------------------------------------------

plot_df = tuning_summary[
    tuning_summary["model"].isin(
        ["RandomForest", "XGBoost"]
    )
].copy()

plot_df["version_clean"] = (
    plot_df["version"]
    .astype(str)
    .str.lower()
    .str.strip()
)

validation_order = [
    "random_5fold",
    "spatial_kmeans_leave_one_out",
]

validation_labels = {
    "random_5fold": "Random CV",
    "spatial_kmeans_leave_one_out": "Spatial CV",
}

series_order = [
    ("RandomForest", "untuned", "RF Untuned"),
    ("RandomForest", "tuned",   "RF Tuned"),
    ("XGBoost",     "untuned", "XGB Untuned"),
    ("XGBoost",     "tuned",   "XGB Tuned"),
]

# Custom colours
series_colors = {
    "RF Untuned": "#A9CBE8",   # light blue
    "RF Tuned":   "#4C78A8",   # blue
    "XGB Untuned": "#F6C28B",  # light orange
    "XGB Tuned":   "#F28E2B",  # orange
}


# ------------------------------------------------------------
# 2. Extract mean R² and SD
# ------------------------------------------------------------

means = {}
stds = {}

for model, version, label in series_order:

    temp = (
        plot_df[
            (plot_df["model"] == model)
            & (plot_df["version_clean"] == version)
        ]
        .set_index("validation")
        .reindex(validation_order)
    )

    means[label] = temp["R2_mean"].to_numpy()
    stds[label] = temp["R2_std"].to_numpy()


# ------------------------------------------------------------
# 3. Check missing rows
# ------------------------------------------------------------

for label in means:

    if np.isnan(means[label]).any():
        print(
            f"Warning: missing values detected for {label}. "
            "Check model/version names in tuning_summary."
        )


# ------------------------------------------------------------
# 4. Plot grouped bars
# ------------------------------------------------------------

x = np.arange(len(validation_order))

bar_width = 0.18

offsets = [
    -1.5 * bar_width,
    -0.5 * bar_width,
     0.5 * bar_width,
     1.5 * bar_width,
]

fig, ax = plt.subplots(
    figsize=(11, 6.5),
    constrained_layout=True,
)

all_bars = []

for offset, (_, _, label) in zip(
    offsets,
    series_order,
):

    bars = ax.bar(
        x + offset,
        means[label],
        width=bar_width,
        yerr=stds[label],
        capsize=4,
        label=label,
        color=series_colors[label],
        edgecolor="black",
        linewidth=0.5,
    )

    all_bars.append(
        (bars, label)
    )


# ------------------------------------------------------------
# 5. Zero reference line
# ------------------------------------------------------------

ax.axhline(
    y=0,
    color="black",
    linewidth=1,
)


# ------------------------------------------------------------
# 6. Automatically calculate safe y-axis limits
# ------------------------------------------------------------

all_lower = []
all_upper = []

for label in means:

    mean_values = means[label]
    std_values = stds[label]

    all_lower.extend(
        mean_values - std_values
    )

    all_upper.extend(
        mean_values + std_values
    )

y_min = np.nanmin(all_lower)
y_max = np.nanmax(all_upper)

y_range = y_max - y_min

if y_range == 0:
    y_range = 1

padding = y_range * 0.18

ax.set_ylim(
    y_min - padding,
    y_max + padding,
)


# ------------------------------------------------------------
# 7. Add mean R² labels
# ------------------------------------------------------------

label_offset = y_range * 0.025

for bars, label in all_bars:

    std_values = stds[label]

    for i, bar in enumerate(bars):

        height = bar.get_height()

        if np.isnan(height):
            continue

        if height >= 0:

            y_text = (
                height
                + std_values[i]
                + label_offset
            )

            va = "bottom"

        else:

            y_text = (
                height
                - std_values[i]
                - label_offset
            )

            va = "top"

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            y_text,
            f"{height:.3f}",
            ha="center",
            va=va,
            fontsize=9,
        )


# ------------------------------------------------------------
# 8. Formatting
# ------------------------------------------------------------

ax.set_xticks(x)

ax.set_xticklabels(
    [
        validation_labels[v]
        for v in validation_order
    ],
    fontsize=11,
)

ax.set_ylabel(
    "Mean outer-fold $R^2$",
    fontsize=11,
)

ax.set_title(
    "Random Forest and XGBoost Before and After Hyperparameter Tuning",
    fontsize=13,
    pad=14,
)

ax.legend(
    ncol=2,
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.00),
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.margins(x=0.10)


# ------------------------------------------------------------
# 9. Save
# ------------------------------------------------------------

figure_path = (
    OUTPUT_DIR
    / "rf_xgb_untuned_vs_tuned_mean_r2.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.15,
)

plt.show()

print(
    f"Saved figure to: {figure_path}"
)